# MidMamba Local Baseline Smoke

Run this notebook from your local Jupyter environment to verify the repo, find a local Databento `.dbn.zst` file, run immediate/TWAP baselines, and smoke-test the execution environment.

The unit tests do not require DBN files. The baseline cells require at least one `.dbn.zst` file under `data/` or a full path assigned to `DBN_FILE`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "midmamba").exists():
            return candidate
    raise RuntimeError("Could not find the midmamba repo root. Start Jupyter from the repo or edit REPO_ROOT manually.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("repo:", REPO_ROOT)
print("python:", sys.executable)

repo: /Users/ak/Documents/genaiexperiments/midmamba
python: /opt/anaconda3/bin/python


In [2]:
def run(cmd, *, check=True):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    env = {**os.environ, "PYTHONPATH": src_path + os.pathsep + os.environ.get("PYTHONPATH", "")}
    proc = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with return code {proc.returncode}")
    return proc.returncode

## 1. Run Unit Tests

In [3]:
run([sys.executable, "-m", "pytest", "tests", "-q"])

$ /opt/anaconda3/bin/python -m pytest tests -q
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: /Users/ak/Documents/genaiexperiments/midmamba
configfile: pyproject.toml
plugins: anyio-4.10.0
collected 28 items

tests/test_baselines.py ....                                             [ 14%]
tests/test_execution_env.py .........                                    [ 46%]
tests/test_lob_mamba.py .....                                            [ 64%]
tests/test_mbp10_features.py ....                                        [ 78%]
tests/test_window_loader.py ......                                       [100%]

============================== 28 passed in 2.33s ==============================



0

## 2. Find a Local DBN File

If nothing is found, set `DBN_FILE = Path("/full/path/to/file.dbn.zst")` in the next cell.

In [4]:
dbn_files = sorted(REPO_ROOT.glob("data/**/*.dbn.zst"))
print(f"found {len(dbn_files)} DBN files")
for path in dbn_files[:10]:
    print(path)

DBN_FILE = dbn_files[0] if dbn_files else None
DBN_FILE

found 42 DBN files
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250304.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250305.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250306.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250307.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250310.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250311.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250312.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250313.mbp-10.dbn.zst
/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250314.mbp-10.dbn.zst


PosixPath('/Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst')

## 3. Baseline Parameters

In [5]:
SAMPLE_ROWS = 10_000
WINDOW_STEPS = 1_000
PARENT_QUANTITY = 10_000
TWAP_SLICES = 20
SIDE = "buy"
OUTPUT_JSON = REPO_ROOT / "results" / "baseline_smoke_local.json"

print("DBN_FILE:", DBN_FILE)
print("output:", OUTPUT_JSON)

DBN_FILE: /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
output: /Users/ak/Documents/genaiexperiments/midmamba/results/baseline_smoke_local.json


## 4. Run Immediate and TWAP Baselines

In [6]:
if DBN_FILE is None:
    raise FileNotFoundError("No DBN file found. Put a .dbn.zst under data/ or set DBN_FILE to a full path.")

run([
    sys.executable,
    "scripts/run_baseline_smoke.py",
    "--dbn-file", DBN_FILE,
    "--sample-rows", SAMPLE_ROWS,
    "--window-steps", WINDOW_STEPS,
    "--parent-quantity", PARENT_QUANTITY,
    "--twap-slices", TWAP_SLICES,
    "--side", SIDE,
    "--output-json", OUTPUT_JSON,
])

$ /opt/anaconda3/bin/python scripts/run_baseline_smoke.py --dbn-file /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst --sample-rows 10000 --window-steps 1000 --parent-quantity 10000 --twap-slices 20 --side buy --output-json /Users/ak/Documents/genaiexperiments/midmamba/results/baseline_smoke_local.json
[baseline] loading /Users/ak/Documents/genaiexperiments/midmamba/data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
[baseline] sample_rows=10000 window_steps=1000 start=0
[baseline] sampled_window_rows=1000 features=138
{
  "immediate": {
    "name": "immediate",
    "total_reward": -495.01261829653,
    "steps": 1,
    "filled_qty": 100.0,
    "remaining_inventory": 9900.0,
    "implementation_shortfall": 7.5000000000045475,
    "implementation_shortfall_bps": 0.012618296529976106,
    "cash": -59445.00000000001,
    "terminal_penalty_bps": 495.0
  },
  "twap": {
    "name": "twap",
    "total_reward": -138.68743217665593,
    "steps": 20,
   

0

In [7]:
report = json.loads(OUTPUT_JSON.read_text())
report["baselines"]

{'immediate': {'name': 'immediate',
  'total_reward': -495.01261829653,
  'steps': 1,
  'filled_qty': 100.0,
  'remaining_inventory': 9900.0,
  'implementation_shortfall': 7.5000000000045475,
  'implementation_shortfall_bps': 0.012618296529976106,
  'cash': -59445.00000000001,
  'terminal_penalty_bps': 495.0},
 'twap': {'name': 'twap',
  'total_reward': -138.68743217665593,
  'steps': 20,
  'filled_qty': 8834.0,
  'remaining_inventory': 1166.0,
  'implementation_shortfall': 47780.27999999988,
  'implementation_shortfall_bps': 80.38743217665595,
  'cash': -5298489.030000001,
  'terminal_penalty_bps': 58.3}}

## 5. Smoke-Test the PPO-Facing Environment

In [8]:
import numpy as np

from midmamba.data import MBP10WindowLoader
from midmamba.env import MidMambaExecutionEnv

loader = MBP10WindowLoader.from_dbn_file(DBN_FILE, sample_rows=min(SAMPLE_ROWS, 5_000), seed=1)
env = MidMambaExecutionEnv(loader, execution_steps=60, initial_inventory=float(PARENT_QUANTITY), side=SIDE)

obs, info = env.reset()
print("obs_shape:", obs.shape)
print("reset_info:", info)

# Try one fully aggressive market action.
next_obs, reward, terminated, truncated, step_info = env.step(np.array([1.0, 1.0], dtype=np.float32))
print("next_obs_shape:", next_obs.shape)
print("reward:", reward)
print("terminated:", terminated, "truncated:", truncated)
step_info

obs_shape: (140,)
reset_info: {'side': 'buy', 'step': 0, 'arrival_price': 594.0899999999999, 'inventory': 10000.0, 'filled_qty': 0.0, 'cash': 0.0, 'executed_shares': 0.0, 'avg_exec_price': 0.0, 'levels_touched': 0, 'implementation_shortfall': 0.0, 'implementation_shortfall_bps': 0.0, 'terminal_penalty_bps': 0.0}
next_obs_shape: (140,)
reward: -15.645777575789452
terminated: True truncated: False


{'side': 'buy',
 'step': 1,
 'arrival_price': 594.0899999999999,
 'inventory': 0.0,
 'filled_qty': 10000.0,
 'cash': -5950195.0,
 'executed_shares': 10000.0,
 'avg_exec_price': 595.0195,
 'levels_touched': 7,
 'implementation_shortfall': 9295.000000000755,
 'implementation_shortfall_bps': 15.645777575789452,
 'terminal_penalty_bps': 0.0}

## Next

If this notebook passes, the next project step is adding the PPO rollout buffer and `train_ppo_smoke.py`.

In [10]:
!python scripts/train_ppo_smoke.py \
  --updates 3 \
  --rollout-steps 128 \
  --execution-steps 60 \
  --seq-len 16 \
  --synthetic-rows 5000 \
  --parent-quantity 1000 \
  --output-json results/ppo_smoke_metrics.json


/opt/anaconda3/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=18500) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[ppo] using synthetic MBP-10 book rows=5000
[ppo] device=cpu backend=gru obs_features=140 seq_len=16 rollout_steps=128 updates=3
[ppo] update=1/3 reward_sum=-1.3309 episodes=4 loss=0.055984 policy=-0.009315 value=0.187363 entropy=2.838257
[ppo] update=2/3 reward_sum=-1.8914 episodes=4 loss=-0.002033 policy=-0.004697 value=0.062128 entropy=2.839957
[ppo] update=3/3 reward_sum=-0.8624 episodes=4 loss=-0.000770 policy=0.001532 value=0.052217 entropy=2.841030
[ppo] wrote /Users/ak/Documents/genaiexperiments/midmamba/results/ppo_smoke_metrics.json


In [11]:
!python scripts/train_ppo_smoke.py \
  --dbn-file data/march2025/xnas-itch-20250303.mbp-10.dbn.zst \
  --chunk-rows 5000 \
  --window-steps 1000 \
  --execution-steps 60 \
  --updates 2 \
  --rollout-steps 128 \
  --seq-len 16 \
  --parent-quantity 1000 \
  --output-json results/ppo_smoke_dbn_metrics.json


/opt/anaconda3/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=18500) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


[ppo] loading data/march2025/xnas-itch-20250303.mbp-10.dbn.zst
[ppo] chunk=1 decoded_rows=5,000 kept_rows=5,000
[ppo] device=cpu backend=gru obs_features=140 seq_len=16 rollout_steps=128 updates=2
[ppo] update=1/2 reward_sum=-12.1162 episodes=5 loss=0.750092 policy=0.000801 value=1.555328 entropy=2.837305
[ppo] update=2/2 reward_sum=-10.8686 episodes=5 loss=0.229205 policy=-0.004573 value=0.524294 entropy=2.836894
[ppo] wrote /Users/ak/Documents/genaiexperiments/midmamba/results/ppo_smoke_dbn_metrics.json


In [ ]:
!python scripts/train_ppo_smoke.py \
  --dbn-file data/march2025/xnas-itch-20250303.mbp-10.dbn.zst \
  --chunk-rows 100000 \
  --max-chunks 80 \
  --rth-only \
  --window-steps 2000 \
  --execution-steps 60 \
  --updates 2 \
  --rollout-steps 128 \
  --seq-len 16 \
  --parent-quantity 1000 \
  --output-json results/ppo_smoke_rth_metrics.json
